# 실전 09-1: Multi-Query Retriever (질문 다각화로 검색 누락 방지)

## 1. 실험 목적
- **문제점**: 사용자가 질문을 너무 짧게 하거나, 문서에 쓰인 단어와 전혀 다른 단어로 질문하면 Vector DB가 관련된 문서를 찾지 못하는 현상이 발생합니다.
- **해결책**: 사용자의 원래 질문을 LLM에게 던져서 **비슷하지만 다른 표현을 가진 3~5개의 질문으로 다각화(Multi-Query)**한 뒤, 각 질문으로 모두 검색을 돌려 결과를 합칩니다.

## 2. 진짜 실무 환경 구축: 실제 영문 PDF 논문 다운로드
장난감 텍스트가 아닌, 그 유명한 트랜스포머 모델 논문 **"Attention Is All You Need"** 원본 PDF를 다운로드하여 RAG를 구축합니다.

In [1]:
import os
import requests
from dotenv import load_dotenv

load_dotenv()

# 1. 데이터 폴더 준비 및 실제 PDF 다운로드
os.makedirs("../data", exist_ok=True)
pdf_path = "../data/attention_is_all_you_need.pdf"
url = "https://arxiv.org/pdf/1706.03762.pdf"

if not os.path.exists(pdf_path):
    print("다운로드 중: Attention Is All You Need 논문 PDF...")
    response = requests.get(url)
    with open(pdf_path, "wb") as f:
        f.write(response.content)
    print("다운로드 완료!")
else:
    print("이미 파일이 존재합니다.")

다운로드 중: Attention Is All You Need 논문 PDF...
다운로드 완료!


## 3. PDF 로드 및 Vector DB 세팅

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# 2. PDF 파싱 및 청킹 (쪼개기)
loader = PyPDFLoader(pdf_path)
pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(pages)
print(f"총 {len(splits)}개의 조각으로 쪼개졌습니다.")

# 3. Vector DB에 저장 (Chroma DB 활용)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(documents=splits, embedding=embeddings)

# 일반 검색기 (Retriever) 생성
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

C:\Users\USER\AppData\Local\Temp\ipykernel_6468\569340101.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


총 52개의 조각으로 쪼개졌습니다.


## 4. Multi-Query Retriever 적용

In [4]:
import logging
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_openai import ChatOpenAI

# 다각화된 쿼리들이 내부적으로 어떻게 생성되는지 눈으로 확인하기 위해 로그 레벨을 조정합니다.
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# MultiQueryRetriever 생성 (base_retriever를 감쌉니다)
mq_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever, 
    llm=llm
)

## 5. 비교 실험: 일반 검색 vs Multi-Query 검색
사용자가 논문의 핵심 개념인 "Transformer architecture"를 묻고 싶지만, 너무 대충 **"What is the model proposed here?"** 라고 물어봤다고 가정해 봅시다.

In [5]:
user_question = "What is the main model proposed here?"

print("\n=== [실험 1] 일반 Retriever 로 검색 ===")
normal_docs = base_retriever.invoke(user_question)
for i, doc in enumerate(normal_docs):
    print(f"[Doc {i+1}] {doc.page_content[:150]}...")

print("\n=== [실험 2] Multi-Query Retriever 로 검색 ===")
print("(LLM이 이 질문을 어떻게 다각화하는지 터미널/콘솔 로그를 확인하세요!)")
mq_docs = mq_retriever.invoke(user_question)
for i, doc in enumerate(mq_docs):
    print(f"[Doc {i+1}] {doc.page_content[:150]}...")


=== [실험 1] 일반 Retriever 로 검색 ===
[Doc 1] mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolu...
[Doc 2] Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalis...
[Doc 3] In this work we propose the Transformer, a model architecture eschewing recurrence and instead
relying entirely on an attention mechanism to draw glob...

=== [실험 2] Multi-Query Retriever 로 검색 ===
(LLM이 이 질문을 어떻게 다각화하는지 터미널/콘솔 로그를 확인하세요!)
[Doc 1] Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalis...
[Doc 2] best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency pars...
[Doc 3] mechanism. We propose a new simple network architecture, the Tr

## 6. 결과 해석

1. **일반 검색의 한계**: 사용자가 "main model"이라는 단어를 썼기 때문에, 벡터 검색은 논문 내용 중 "model"이라는 단어가 많이 들어간 엉뚱한 페이지나 수식을 가져올 확률이 높습니다.
2. **Multi-Query의 위력**: 위에서 로그로 출력된 내용을 보면, LLM이 사용자의 질문을 다음과 같이 똑똑하게 바꿉니다.
   - "What is the primary architecture introduced in this document?"
   - "Which model is the core focus of this paper?"
   이처럼 다양한 표현으로 여러 번 검색한 뒤, 겹치는 문서들을 통합해서 가장 관련성 높은 문서를 최종적으로 뽑아줍니다. 이로 인해 검색 누락(False Negative)이 크게 줄어듭니다.